# 03 · Composition Basics

This notebook introduces USD's composition system — the technology that makes collaborative, non-destructive, and modular 3D workflows possible. You will combine, layer, and reference scene elements so you can build projects that scale from a single asset up to a full virtual world.

## Objectives

By the end of this notebook you will be able to:

- **Understand layers and composition** — see how USD combines multiple data sources into one coherent scene.
- **Navigate LIVRPS strength ordering** — learn the rules that determine which opinion wins when conflicts arise.
- **Use specifiers effectively** — control how prims are interpreted with `def`, `over`, and `class`.
- **Create modular assets with references** — build reusable components that can be shared between projects.
- **Set default prims** — establish clear entry points for composed assets.
- **Implement variant sets** — create interchangeable variations of an asset that can be swapped at runtime.

## How to use this notebook

Run the two bootstrap cells at the top first — they check your environment and load helper utilities (with local fallbacks if `lousd` is not installed). Then walk through each lesson in order. Every lesson mirrors one chapter of the `docs/composition-basics/` module: a short summary, hands-on code, a preview of the authored USD, and a brief debrief. All files are written under `./\_assets/` so you can inspect the authored USD directly on disk.

In [1]:
import sys
print("Python:", sys.version.split()[0])
from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK.")
from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9


pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/03_composition_basics/_assets


In [2]:
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


## 3.1  Layers

USD scenes are organized into **layers** — individual files or resources that each hold a subset of the total scene description (geometry, materials, animation, overrides, and so on). Layers compose together at runtime into a single coherent stage without destructive merges, so multiple departments can edit the same shot in parallel. A layer is usually a `.usd`, `.usda`, or `.usdc` file, but it can also come from a plugin format or even a database. Each stage is backed by a **layer stack** that USD composes via the LIVRPS rules you will meet in the next lesson.

**Key APIs**
- `Usd.Stage.CreateNew(path)` / `Usd.Stage.Open(path)` — create or open a stage rooted at a layer.
- `stage.GetRootLayer()` — access the stage's root `Sdf.Layer`.
- `rootLayer.subLayerPaths` — list of sublayers (weaker than the root itself).
- `layer.Save()` / `layer.Reload()` — persist or discard unsaved opinions.

Docs: [`../docs/composition-basics/layers.md`](../docs/composition-basics/layers.md)

The source lesson has no executable cells, so we build our own small demo: create a base layer and a stronger layer that sublayers it.

In [3]:
from pxr import Usd, UsdGeom, Sdf

# Base layer: define a sphere with radius 1.
base_path = "_assets/layers_base.usda"
base_stage = create_new_stage(base_path)
sphere = UsdGeom.Sphere.Define(base_stage, "/World/Ball")
sphere.GetRadiusAttr().Set(1.0)
UsdGeom.Xform.Define(base_stage, "/World")
base_stage.SetDefaultPrim(base_stage.GetPrimAtPath("/World"))
base_stage.Save()

# Strong layer: sublayer the base and override the radius non-destructively.
strong_path = "_assets/layers_strong.usda"
strong_stage = create_new_stage(strong_path)
strong_stage.GetRootLayer().subLayerPaths = ["layers_base.usda"]

# Author an 'over' on the same prim path and change the radius.
strong_stage.OverridePrim("/World/Ball")
UsdGeom.Sphere.Get(strong_stage, "/World/Ball").GetRadiusAttr().Set(5.0)
strong_stage.Save()

print("Sublayers on strong layer:", list(strong_stage.GetRootLayer().subLayerPaths))
print("Composed radius:", UsdGeom.Sphere.Get(strong_stage, "/World/Ball").GetRadiusAttr().Get())

Sublayers on strong layer: ['layers_base.usda']
Composed radius: 5.0


In [4]:
DisplayUSD("_assets/layers_strong.usda", show_usd_code=True)

**What just happened:** the base layer defined the sphere with radius `1`, and the strong layer — which sublayers the base — authored an `over` that raises the radius to `5`. Because the strong layer's opinion is, well, stronger, the composed stage reports `5.0`. The original file on disk is untouched: this is non-destructive editing in action.

## 3.2  Strength Ordering (LIVRPS)

When multiple layers speak about the same prim or property, USD has to decide which opinion wins. Strength ordering is the deterministic rule set that makes that decision, and it is usually abbreviated **LIVRPS** (pronounced "liver peas"): **L**ocal, **I**nherits, **V**ariant sets, **R**eferences, **P**ayloads, **S**pecializes — strongest to weakest. Local opinions authored directly on the root layer stack always win; specializes acts as a fallback. LIVRPS is applied recursively, so opinions inside a reference are resolved by their own local LIVRPS before that composed result slots into the outer stage.

**Key APIs**
- `prim.GetPrimStack()` — list of `SdfPrimSpec` contributing to a prim, ordered strongest to weakest.
- `prim.GetInherits()`, `prim.GetReferences()`, `prim.GetPayloads()`, `prim.GetSpecializes()` — per-arc edit accessors.
- `attr.GetPropertyStack(time)` — ordered list of attribute opinions at a given time.

Docs: [`../docs/composition-basics/strength-ordering.md`](../docs/composition-basics/strength-ordering.md)

This lesson has no executable cells in source. We reuse the two layers from 3.1 and print the prim stack so LIVRPS becomes visible.

In [5]:
from pxr import Usd, UsdGeom

# Re-open the stronger layer authored in 3.1.
stage = Usd.Stage.Open("_assets/layers_strong.usda")
ball = stage.GetPrimAtPath("/World/Ball")

print("Composed radius (winner):", UsdGeom.Sphere(ball).GetRadiusAttr().Get())
print()
print("Prim stack for", ball.GetPath(), "(strong -> weak):")
for spec in ball.GetPrimStack():
    print("  -", spec.layer.identifier.split('/')[-1], "specifier=", spec.specifier)

print()
print("Property stack for radius (strong -> weak):")
for spec in UsdGeom.Sphere(ball).GetRadiusAttr().GetPropertyStack(Usd.TimeCode.Default()):
    print("  -", spec.layer.identifier.split('/')[-1], "value=", spec.default)

Composed radius (winner): 5.0

Prim stack for /World/Ball (strong -> weak):
  - layers_strong.usda specifier= Sdf.SpecifierOver
  - layers_base.usda specifier= Sdf.SpecifierDef

Property stack for radius (strong -> weak):
  - layers_strong.usda value= 5.0
  - layers_base.usda value= 1.0


**What just happened:** the prim stack shows both layers contributing opinions, with the root (strong) layer listed first. The property stack for `radius` confirms the strong layer contributes `5.0` and the base sublayer still contributes `1.0`, but the strong opinion wins per LIVRPS's local rule.

## 3.3  Specifiers

Every prim has a **specifier** that conveys how the prim should be interpreted when composed. `def` ("define") concretely declares a prim and is the only specifier visited by default stage traversals such as rendering. `over` ("override") sparsely modifies opinions for a prim that is defined elsewhere, enabling non-destructive edits. `class` marks an abstract blueprint prim that is meant to be composed onto other prims via inherits, references, payloads, or specializes — it is present on the stage but ignored by default traversals.

**Key APIs**
- `prim.GetSpecifier()` / `prim.SetSpecifier(Sdf.SpecifierDef|Over|Class)`
- `stage.CreateClassPrim(path)` — convenience for authoring a class prim.
- `stage.OverridePrim(path)` — author an `over` for the given path.
- `prim.GetInherits().AddInherit(path)` — compose a class onto a concrete prim.

Docs: [`../docs/composition-basics/specifiers.md`](../docs/composition-basics/specifiers.md)

### Example 1: Authoring defs and a class in a base layer

This script defines two concrete cubes (specifier `def`) and a reusable `class` prim that holds a `displayColor` opinion for inheritance. The output contrasts the composed specifiers, showing `def` for scene geometry and `class` for the abstract prim.

In [6]:
from pxr import Usd, UsdGeom, Sdf, Gf

base_file_path = "_assets/specifiers_base.usda"

stage = create_new_stage(base_file_path)

# Create a simple scene with two cubes
world = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())

box_prim_cube = UsdGeom.Cube.Define(stage, world.GetPath().AppendChild("Box"))
box_prim_cube.GetSizeAttr().Set(2.0)

box_2_prim_cube = UsdGeom.Cube.Define(stage, world.GetPath().AppendChild("Box_2"))
box_2_prim_cube.GetSizeAttr().Set(2.0)
box_2_api = UsdGeom.XformCommonAPI(box_2_prim_cube)
box_2_api.SetTranslate(Gf.Vec3d(5, 0, 0))

# Define a class prim with a displayColor primvar
class_prim = stage.CreateClassPrim(world.GetPath().AppendPath("_Look/_green"))
class_prim_primvar_api = UsdGeom.PrimvarsAPI(class_prim)
class_prim_primvar_api.CreatePrimvar("displayColor", Sdf.ValueTypeNames.Color3fArray).Set([Gf.Vec3f(0.1, 0.8, 0.2)])

# Inspect specifiers
print("box specifier:", box_prim_cube.GetPrim().GetSpecifier())  # Sdf.SpecifierDef
print("box_2 specifier:", box_2_prim_cube.GetPrim().GetSpecifier())  # Sdf.SpecifierDef
print("class_prim specifier:", class_prim.GetSpecifier())  # e.g. Sdf.SpecifierClass

stage.Save()

box specifier: Sdf.SpecifierDef
box_2 specifier: Sdf.SpecifierDef
class_prim specifier: Sdf.SpecifierClass


In [7]:
DisplayUSD("_assets/specifiers_base.usda", show_usd_code=True)

**What just happened:** both `Box` and `Box_2` were authored with `Sdf.SpecifierDef` so they are visible to traversals, while `/World/_Look/_green` was authored as a `class` — it lives on the stage as a blueprint holding the green `displayColor` but default traversals will skip it.

### Example 2: Non-destructive `over` edits and a class inherit

This script sublayers the base file, authors an `over` on `/World/Box` to change `size`, and adds an **inherits** arc so the box picks up the class color. The prim-stack printout makes it clear that the strong layer contributes `over` while the base contributes `def`, and the class opinions are applied via the inherit.

In [8]:
from pxr import Usd, UsdGeom, Sdf

new_file_path = "_assets/specifiers_over_base.usda"
base_file_path = "specifiers_base.usda"  # path relative to the new file

stage = create_new_stage(new_file_path)

# Base is weaker than the root layer (root opinions are strongest)
stage.GetRootLayer().subLayerPaths = [base_file_path]

prim_path = "/World/Box"

# Author an override for the same prim
stage.OverridePrim(prim_path)

# Get the cube and change its size
box_prim_cube = UsdGeom.Cube.Get(stage, prim_path)
box_prim_cube.GetSizeAttr().Set(4.0)

# Compose the class onto the box using an inherit arc
box_prim_cube.GetPrim().GetInherits().AddInherit(Sdf.Path("/World/_Look/_green"))

# SdfPrimSpec handles, ordered strong to weak
for prim_spec in box_prim_cube.GetPrim().GetPrimStack():
    print(" - layer:", prim_spec.layer.identifier.split('/')[-1], "specifier:", prim_spec.specifier)

stage.Save()

 - layer: specifiers_over_base.usda specifier: Sdf.SpecifierOver
 - layer: specifiers_base.usda specifier: Sdf.SpecifierDef
 - layer: specifiers_base.usda specifier: Sdf.SpecifierClass


In [9]:
DisplayUSD("_assets/specifiers_over_base.usda", show_usd_code=True)

**What just happened:** the prim stack shows two entries for `/World/Box` — the strong layer contributing `SpecifierOver` and the base sublayer contributing `SpecifierDef`. Size is overridden to `4.0`, and the inherit arc pulls the green `displayColor` opinions from the class, all without touching the original base file.

## 3.4  References

A **reference** is a composition arc that grafts a prim (and its descendants) from another layer onto a prim in the current stage. References are how USD scales: instead of duplicating data, you compose small modular assets into larger assemblies, and any edits in the destination stage become non-destructive overrides on top of the referenced data. A reference arc carries a layer identifier and, optionally, a prim path — the prim path can be omitted when the referenced layer has a `defaultPrim` set. References are the second most important arc after sublayers.

**Key APIs**
- `prim.GetReferences()` — returns a `UsdReferences` object.
- `references.AddReference(assetPath, primPath=None)` — add an external or internal reference.
- `references.ClearReferences()` — remove all references from a prim.

Docs: [`../docs/composition-basics/references.md`](../docs/composition-basics/references.md)

### Example 1: Adding a reference

In [10]:
from pxr import Usd, UsdGeom, Gf

# Create a new stage and define a cube:
file_path = "_assets/cube.usda"
stage = create_new_stage(file_path)
cube = UsdGeom.Cube.Define(stage, "/Cube")
stage.SetDefaultPrim(cube.GetPrim())
stage.Save()

# Create a second file path and stage, define a world and a sphere:
second_file_path = "_assets/shapes.usda"
stage = create_new_stage(second_file_path)
world = UsdGeom.Xform.Define(stage, "/World")
UsdGeom.Sphere.Define(stage, world.GetPath().AppendPath("Sphere"))

# Define a reference prim and set its translation:
reference_prim = stage.DefinePrim(world.GetPath().AppendPath("Cube_Ref"))

# Add a reference to the "cube.usda" file:
reference_prim.GetReferences().AddReference("./cube.usda")
# Position the cube
UsdGeom.XformCommonAPI(reference_prim).SetTranslate(Gf.Vec3d(5, 0, 0))

stage.Save()

In [11]:
DisplayUSD("_assets/shapes.usda", show_usd_code=True)

**What just happened:** we saved a standalone `cube.usda` with a `defaultPrim`, then authored `shapes.usda` which references that file at `/World/Cube_Ref` and translates the referenced cube. Because `cube.usda` has a default prim, we did not need to name a target prim path on the reference.

### Example 2: Referencing an external asset

Next we reference a shipping box asset that ships with the repository under `docs/exercise_content/foundations/cubebox_a02`. The setup cell copies it next to our notebook; if the path is different in your checkout, adjust `src`.

In [12]:
import shutil
from pathlib import Path

# Search broadly for the cubebox asset by walking up from cwd looking for a repo root with docs/exercise_content/
def _find_cubebox():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / "docs" / "exercise_content" / "foundations" / "cubebox_a02"
        if candidate.exists():
            return candidate
        # also try sibling (notebooks/ case)
        candidate = base.parent / "docs" / "exercise_content" / "foundations" / "cubebox_a02"
        if candidate.exists():
            return candidate
    return None

src = _find_cubebox()
dest = Path("_assets/cubebox_a02")
if src is None:
    print("WARNING: cubebox_a02 exercise asset not found. Downstream reference cells will be skipped.")
    HAVE_CUBEBOX = False
elif dest.exists():
    print(f"Asset already staged at {dest}")
    HAVE_CUBEBOX = True
else:
    shutil.copytree(src, dest)
    print(f"Copied asset from {src}")
    HAVE_CUBEBOX = True

Copied asset from /Users/stranzersweb/Projects/LearnOpenUSD/docs/exercise_content/foundations/cubebox_a02


In [13]:
if HAVE_CUBEBOX:
    from pxr import Usd, UsdGeom

    file_path = "_assets/asset_ref.usda"
    stage: Usd.Stage = create_new_stage(file_path)

    # Define a root Xform named "World"
    world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

    # Define a child Xform named "Geometry" under the "World" Xform
    geometry_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, world_xform.GetPath().AppendPath("Geometry"))

    # Define a new Xform named "Box" under the root "Geometry" Xform
    box_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, geometry_xform.GetPath().AppendPath("Box"))
    box_prim: Usd.Prim = box_xform.GetPrim()

    # Add a reference to a USD file containing a box geometry
    box_prim.GetReferences().AddReference("./cubebox_a02/cubebox_a02.usd")

    stage.Save()
else:
    print("Skipping — cubebox asset not available.")

In [14]:
if HAVE_CUBEBOX:
    DisplayUSD("_assets/asset_ref.usda", show_usd_code=True)
else:
    print("Skipping — cubebox asset not available.")

**What just happened:** `asset_ref.usda` is a small assembly layer that brings in the `cubebox_a02` asset under `/World/Geometry/Box`. The referenced asset's prims and properties compose onto the Box prim without copying geometry — that is how production pipelines reuse libraries of props, characters, and set dressing at scale.

## 3.5  Default Prim

A **default prim** is a top-level prim recorded in the layer's metadata as the primary entry point for the stage. When another stage references or payloads your layer and does not specify a target prim path, USD uses the default prim. Setting one is a best practice — `usdchecker` will flag its absence — and it makes your asset self-describing when composed elsewhere. The default prim must be a top-level prim of the layer.

**Key APIs**
- `stage.SetDefaultPrim(prim)` / `stage.GetDefaultPrim()`
- `stage.HasDefaultPrim()` / `stage.ClearDefaultPrim()`
- `rootLayer.defaultPrim` — the underlying metadata field on `Sdf.Layer`.

Docs: [`../docs/composition-basics/default-prim.md`](../docs/composition-basics/default-prim.md)

### Example 1: Setting a default prim

In [15]:
from pxr import Usd

file_path = "_assets/default_prim.usda"
stage: Usd.Stage = create_new_stage(file_path)
stage.DefinePrim("/hello")
stage.DefinePrim("/hello/world")
hello_prim: Usd.Prim = stage.GetPrimAtPath("/hello")

# Set the default primitive of the stage to the primitive at "/hello":
stage.SetDefaultPrim(hello_prim)

stage.Save()

In [16]:
DisplayCode("_assets/default_prim.usda")

**What just happened:** the authored USDA now carries `defaultPrim = "hello"` in its layer metadata. Any downstream stage that references `default_prim.usda` without providing a prim path will compose `/hello` by default.

## 3.6  Variant Sets

A **variant set** lets a prim carry several alternative representations — model shapes, material looks, LODs — and switch between them without duplicating data. Each variant set has a name, one or more variants (named choices), and an optional selection. You author opinions inside a variant by entering a **variant edit context**: anything you author while that context is open only applies when that variant is active. Variant selections can be authored on the defining layer or on a downstream layer, so each reference can pick a different option. Strength ordering still applies — variant sets sit in the middle of LIVRPS.

**Key APIs**
- `prim.GetVariantSets()` and `vsets.AddVariantSet("name")`
- `vset.AddVariant("ChoiceA")`, `vset.SetVariantSelection("ChoiceA")`
- `with vset.GetVariantEditContext(): ...` — author inside a variant.
- `vset.GetVariantNames()`, `vset.GetVariantSelection()`, `vset.ClearVariantSelection()`

Docs: [`../docs/composition-basics/variant-sets.md`](../docs/composition-basics/variant-sets.md)

The source lesson has no executable cells, so we build our own demo: a cube prim with a `shadingVariant` set offering `red` and `blue` variants that each author a different `displayColor`.

In [17]:
from pxr import Usd, UsdGeom, Sdf, Gf

file_path = "_assets/variant_sets.usda"
stage = create_new_stage(file_path)

world = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())

cube = UsdGeom.Cube.Define(stage, "/World/Block")
cube.GetSizeAttr().Set(2.0)
block_prim = cube.GetPrim()

# Create a variant set named 'shadingVariant' with two choices.
vset = block_prim.GetVariantSets().AddVariantSet("shadingVariant")

for name, color in (("red", Gf.Vec3f(0.9, 0.1, 0.1)), ("blue", Gf.Vec3f(0.1, 0.3, 0.9))):
    vset.AddVariant(name)
    vset.SetVariantSelection(name)
    with vset.GetVariantEditContext():
        # Opinions authored here only apply when this variant is selected.
        UsdGeom.PrimvarsAPI(block_prim).CreatePrimvar(
            "displayColor", Sdf.ValueTypeNames.Color3fArray
        ).Set([color])

# Pick the initial selection.
vset.SetVariantSelection("red")
print("Variant names:", vset.GetVariantNames())
print("Current selection:", vset.GetVariantSelection())

# Flip to the other variant and read the composed color.
displayColor_attr = UsdGeom.PrimvarsAPI(block_prim).GetPrimvar("displayColor").GetAttr()
print("displayColor when red:", displayColor_attr.Get())
vset.SetVariantSelection("blue")
print("displayColor when blue:", displayColor_attr.Get())

# Leave selection on 'red' for the preview.
vset.SetVariantSelection("red")
stage.Save()

Variant names: ['blue', 'red']
Current selection: red
displayColor when red: [(0.9, 0.1, 0.1)]
displayColor when blue: [(0.1, 0.3, 0.9)]


In [18]:
DisplayUSD("_assets/variant_sets.usda", show_usd_code=True)

**What just happened:** the `Block` prim now owns a `shadingVariant` set with `red` and `blue` choices. Opinions for `displayColor` were authored inside each variant using `GetVariantEditContext()`, so switching the selection swaps the composed color without duplicating the cube. In the USDA output you can see the two variant blocks nested under the prim and the `variantSet` selection recorded as metadata.

## Key Takeaways

- **Layers are the unit of composition.** USD combines multiple layers into a single coherent stage non-destructively, so teams can edit in parallel and overrides never mutate source data.
- **LIVRPS decides the winner.** Local, Inherits, Variant sets, References, Payloads, Specializes — strongest to weakest. Use `prim.GetPrimStack()` and `attr.GetPropertyStack()` to debug who contributed what.
- **Specifiers declare intent.** `def` for real scene content, `over` for non-destructive edits on prims defined elsewhere, `class` for reusable blueprints that are composed via inherit / reference arcs.
- **References scale your pipeline.** Modular assets, referenced once and overridden locally, are how USD handles thousands of props without copying data.
- **Always set a default prim** on assets that will be referenced, so consumers don't have to know your internal hierarchy.
- **Variant sets** give a single prim multiple named representations — shapes, looks, LODs — with opinions isolated to each choice and switchable from downstream layers.

## Next up → `04_beyond_basics.ipynb`

In the next notebook we move past the basics: payloads for deferred loading, specializes for material libraries, instancing, and specialized composition workflows for production-scale scenes.